In [1]:
%pip install --upgrade --quiet  langchain langchain-openai langchain-experimental presidio-analyzer presidio-anonymizer spacy Faker

Note: you may need to restart the kernel to use updated packages.


In [2]:
text = """
""Admission Date:  [**2118-6-2**]       Discharge Date:  [**2118-6-14**]

Date of Birth:                    Sex:  F

Service:  MICU and then to [**Doctor Last Name **] Medicine

HISTORY OF PRESENT ILLNESS: DATE: [**2118-6-14**]  This is an 81-year-old female
with a history of emphysema (not on home O2), who presents
with three days of shortness of breath thought by her primary
care doctor to be a COPD flare.  Two days prior to admission,
she was started on a prednisone taper and one day prior to
admission she required oxygen at home in order to maintain
oxygen saturation greater than 90%.  She has also been on
levofloxacin and nebulizers, and was not getting better, and
presented to the [**Hospital1 18**] Emergency Room.



"""

In [3]:
from presidio_anonymizer import AnonymizerEngine
from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer, RecognizerRegistry
from presidio_anonymizer.entities import OperatorConfig
from faker import Faker
import random
import copy


fake = Faker()
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

name_pool = ["name", "patient", "person"]
hospital_pool = ["hospital", "institution", "clinic"]
date_pool = ["date", "day", "occasion"]
location_pool = ["address", "location", "street"]
month_pool = ["month"]
country_pool = ["country", "land", "nation"]
phone_pool = ["phone", "telephone number", "contact"]
state_pool = ["state", "province", "region"]
year_pool = ["year", "time", "period"]
age_pool = ["age", "lifetime", "seniority"]
def fake_name(text: str, **kwargs) -> str:
    return random.choice(name_pool) + " " + fake.last_name()

def fake_firstName(text: str, **kwargs) -> str:
    return random.choice(name_pool) + " " + fake.first_name()

def fake_hospital(text: str, **kwargs) -> str:
    return random.choice(hospital_pool) + " " + fake.company()

def fake_hospitalUnit(text: str, **kwargs) -> str:
    return random.choice(["Oncology unit", "Critical Care unit", "Psychiatry unit", "Pulmonary unit", "Cardiology unit", "Ginecology unit"])

def fake_date(text: str, **kwargs) -> str:
    return random.choice(date_pool) + " " + str(fake.date_this_decade())

def fake_location(text: str, **kwargs) -> str:
    return random.choice(location_pool) + " " + str(fake.street_address())

def fake_number(text: str, **kwargs) -> str:
    return str(random.randint(1, 1000))

def fake_phone(text: str, **kwargs) -> str:
    return random.choice(phone_pool) + " " + str(fake.phone_number())

def fake_month(text: str, **kwargs) -> str:
    return random.choice(month_pool) + " " + fake.month_name()

def fake_country(text: str, **kwargs) -> str:
    return random.choice(country_pool) + " " + fake.country()

def fake_state(text: str, **kwargs) -> str:
    return random.choice(state_pool) + " " + fake.state()

def fake_year(text: str, **kwargs) -> str:
    return random.choice(year_pool) + " " + str(random.randint(2000, 2022))

def fake_over90_age(text: str, **kwargs) -> str:
    return random.choice(age_pool) + " " + str(random.randint(90, 110))

# Regex patterns that trigger stub functions
firstName_pattern1 = Pattern(
    name="firstNameData",
    regex="\[\*\*First Name\d* \(\w+( \w+)*\)( \d+)?\s?\*\*\]"
,
    score=0.8
)
firstName_pattern2 = Pattern(
    name="firstNameData",
    regex="\[\*\*Initials \(NamePattern\w+\)\s?\d*\*\*\]"
,
    score=0.8
)
name_pattern1 = Pattern(
    name="doctorData",
    regex="\[\*\*\s*Doctor\s+[\w\s]+?\s*\*\*\]"
,  
    score=0.8
)
name_pattern2 = Pattern(
    name="doctorData",
    regex="\[\*\*First\s+Name\d+\s\([A-Za-z]+\)\s\d{4}\*\*\]"
,  
    score=0.8
)
name_pattern3 = Pattern(
    name="doctorData",
    regex="\[\*\*\s*Known\s+lastname(\s+\d+)?\s*\*\*\]"
,  
    score=0.8
)
name_pattern4 = Pattern(
    name="doctorData",
    regex="\[\*\*Last Name \(\w+\)( \d+)?\s?\*\*\]"

,  
    score=0.8
)
overNinetyage_pattern = Pattern(
    name="ageData",
    regex="\[\*\*Age over 90 \*\*\]"

,  
    score=0.8
)
name_pattern5 = Pattern(
    name="doctorData",
    regex="\[\*\*Name\d* \(Name(?:Is|NI)?\)( \d+)?\*\*\]"

,  
    score=0.8
)
name_pattern6 = Pattern(
    name="doctorData",
    regex="\[\*\*Name \(NI\)( \d+)?\*\*\]"

,  
    score=0.8
)


name_pattern7 = Pattern(
    name="doctorData",
    regex="\[\*\*Name \(STitle\) \*\*\]"

,  
    score=0.8
)
name_pattern8 = Pattern(
    name="doctorData",
    regex="\[\*\*Name\d* \(NI\)( \d+|\s*\*\*)\]"

,  
    score=0.8
)
name_pattern9 = Pattern(
    name="doctorData",
    regex="\[\*\*Name\d* \(NI\)( \d+)?\*\*\]"


,  
    score=0.8
)
genericName_pattern = Pattern(
    name="doctorData",
    regex="\[\*\*\s*\w+\s*\(\s*Name\w*\s*\)\s*\*\*\]"


,

    score=0.8
)

genericName_pattern2 = Pattern(
    name="doctorData",
    regex="\[\*\*Name\d*\s*(\([^)]*\))?\s*\d*\s*\*\*\]"

, 

    score=0.8
)
hospital_pattern = Pattern(
    name="hospitalData",
    regex="\[\*\*Hospital\s?\d+(\s\d+)?\s*\*\*\]"
,  
    score=0.8
)
hospital_pattern2 = Pattern(
    name="hospitalData",
    regex="\[\*\*Hospital\s*\*\*\]"
,  
    score=0.8
)
hospitalUnit_pattern = Pattern(
    name="hospitalUnitData",
    regex="\[\*\*\s*Hospital\s+[\w\s]+?\s+\d+\s*\*\*\]"
,  
    score=0.8
)

date_pattern1 = Pattern(
    name="dateData",
    regex="\[\*\*\d{4}-\d{1,2}-\d{1,2}\*\*\]"
,  
    score=0.8
)
date_pattern2 = Pattern(
    name="dateData",
    regex="\[\*\*\d{1,2}-\d{1,2}\*\*\]"
,  
    score=0.8
)
location_pattern1 = Pattern(
    name="locationData",
    regex="\[\*\*Location\s*(\([^)]*\))?\s*\d*\s*\*\*\]"

,  
    score=0.8
)
location_pattern2 = Pattern(
    name="locationData",
    regex="\[\*\*Street Address\(\d\) \d+\*\*\]"

,  
    score=0.8
)
generic_location = Pattern(
    name="locationData",
    regex="\[\*\*.*?Address.*?\*\*\]"

,  
    score=0.8
)

number_pattern1 = Pattern(
    name="numberData",
    regex="\[\*\*\d+\*\*\]"

,  
    score=0.8
)
number_pattern2 = Pattern(
    name="numberData",
    regex="\[\*\*Job Number \d+\*\*\]"

,  
    score=0.8
)
number_pattern3 = Pattern( 
    name="numberData",
    regex="\[\*\*(\w+\s*)?Number\d*\s*(\([^)]*\))?\s*\d*\s*\*\*\]"

,  
    score=0.8
)
telephone_pattern = Pattern(
    name="phoneData",
    regex="\[\*\*Telephone\/Fax \(\d+\) \d*\*\*\]"
,  
    score=0.8
) 
month_pattern = Pattern(
    name="monthData",
    regex="\[\*\*Month \(only\)?\s?\d*\*\*\]"
,  
    score=0.8
)
month_pattern2 = Pattern(
    name="monthData",
    regex="\[\*\*.*?Month.*?\*\*\]"
,  
    score=0.8
)
country_pattern = Pattern(
    name="countryData",
    regex="\[\*\*Country(?: \d+)?\s*\*\*\]"
,  
    score=0.8
)
genericNumber_pattern = Pattern(
    name="numberData",
    regex="\[\*\*\s*Num[a-zA-Z\s]*\s*\*\*\]"
,  
    score=0.8
)
genericNumber_pattern2 = Pattern(
    name="numberData",
    regex="\[\*\*\s*Numeric\s+Identifier(\s+\d+)?\s*\*\*\]"
,  
    score=0.8
)
state_pattern = Pattern(
    name="stateData",
    regex="\[\*\*\s*State(\s+\d+)?\s*\*\*\]"
,  
    score=0.8
)
year_pattern = Pattern(
    name="yearData",
    regex="\[\*\*\s*Year(?:\s*\(\d+\s*digits\))?(\s+\d+)?\s*\*\*\]"
,  
    score=0.8
)
year_recognizer = PatternRecognizer(
    supported_entity="yearData",
    patterns=[year_pattern]
)
state_recognizer = PatternRecognizer(
    supported_entity="stateData",
    patterns=[state_pattern]
)
overNinetyage_recognizer = PatternRecognizer(
    supported_entity="ageData",
    patterns=[overNinetyage_pattern]
)
doctor_recognizer = PatternRecognizer(
    supported_entity="doctorData",
    patterns=[name_pattern1, name_pattern2, name_pattern3, name_pattern4,
               name_pattern5, name_pattern6, name_pattern7,name_pattern8,
                 name_pattern9, genericName_pattern, genericName_pattern2]
)
hospital_recognizer = PatternRecognizer(
    supported_entity="hospitalData",
    patterns=[hospital_pattern, hospital_pattern2]
)
hospitalUnit_recognizer = PatternRecognizer(
    supported_entity="hospitalUnitData",
    patterns=[hospitalUnit_pattern]
)
date_recognizer = PatternRecognizer(
    supported_entity="dateData",
    patterns=[date_pattern1, date_pattern2]
)
location_recognizer = PatternRecognizer(
    supported_entity="locationData",
    patterns=[location_pattern1, location_pattern2, generic_location]
)
number_recognizer = PatternRecognizer(
    supported_entity="numberData",
    patterns=[number_pattern1, number_pattern2, number_pattern3, genericNumber_pattern, genericNumber_pattern2]
)
firstName_recognizer = PatternRecognizer(
    supported_entity="firstNameData",
    patterns=[firstName_pattern1, firstName_pattern2]
)
phone_recognizer = PatternRecognizer(
    supported_entity="phoneData",
    patterns=[telephone_pattern]
)
month_recognizer = PatternRecognizer(
    supported_entity="monthData",
    patterns=[month_pattern, month_pattern2]
)
country_recognizer = PatternRecognizer(
    supported_entity="countryData",
    patterns=[country_pattern]
)
genericNumber_recognizer = PatternRecognizer(
    supported_entity="genericNumberData",
    patterns=[genericNumber_pattern]
)

analyzer.registry.remove_recognizer("NUMBER") 
analyzer.registry.remove_recognizer("US_DRIVER_LICENSE")
analyzer.registry.remove_recognizer("PERSON")

analyzer.registry.add_recognizer(doctor_recognizer)
analyzer.registry.add_recognizer(hospital_recognizer)
analyzer.registry.add_recognizer(hospitalUnit_recognizer)
analyzer.registry.add_recognizer(date_recognizer)
analyzer.registry.add_recognizer(location_recognizer)
analyzer.registry.add_recognizer(number_recognizer)
analyzer.registry.add_recognizer(firstName_recognizer)
analyzer.registry.add_recognizer(phone_recognizer)
analyzer.registry.add_recognizer(month_recognizer)
analyzer.registry.add_recognizer(country_recognizer)
analyzer.registry.add_recognizer(genericNumber_recognizer)
analyzer.registry.add_recognizer(state_recognizer)
analyzer.registry.add_recognizer(year_recognizer)
analyzer.registry.add_recognizer(overNinetyage_recognizer)


staticOperators = {
    "doctorData": OperatorConfig("custom", {"lambda": fake_name}),
    "hospitalData": OperatorConfig("custom", {"lambda": fake_hospital}),
    "hospitalUnitData": OperatorConfig("custom", {"lambda": fake_hospitalUnit}),
    "dateData": OperatorConfig("custom", {"lambda": fake_date}),
    "locationData": OperatorConfig("custom", {"lambda": fake_location}),
    "numberData": OperatorConfig("custom", {"lambda": fake_number}),
    "firstNameData": OperatorConfig("custom", {"lambda": fake_firstName}),
    "phoneData": OperatorConfig("custom", {"lambda": fake_phone}),
    "monthData": OperatorConfig("custom", {"lambda": fake_month}),
    "countryData": OperatorConfig("custom", {"lambda": fake_country}),
    "genericNumberData": OperatorConfig("custom", {"lambda": fake_number}),
    "stateData": OperatorConfig("custom", {"lambda": fake_state}),
    "yearData": OperatorConfig("custom", {"lambda": fake_year}),
    "ageData": OperatorConfig("custom", {"lambda": fake_over90_age}),	
}
staticEntities = ["doctorData", "hospitalData", "hospitalUnitData",
                    "dateData", "locationData", "numberData", "firstNameData",
                      "phoneData", "monthData", "countryData","genericNumberData",
                      "stateData", "yearData", "ageData"]


dynamicEntities = []

dynamicOperators = {}


dynamicAnalyzer = AnalyzerEngine()

staticAnalyzer = AnalyzerEngine()
staticAnalyzer.registry.remove_recognizer("NUMBER") 
staticAnalyzer.registry.remove_recognizer("US_DRIVER_LICENSE")
staticAnalyzer.registry.remove_recognizer("PERSON")

staticAnalyzer.registry.add_recognizer(doctor_recognizer)
staticAnalyzer.registry.add_recognizer(hospital_recognizer)
staticAnalyzer.registry.add_recognizer(hospitalUnit_recognizer)
staticAnalyzer.registry.add_recognizer(date_recognizer)
staticAnalyzer.registry.add_recognizer(location_recognizer)
staticAnalyzer.registry.add_recognizer(number_recognizer)
staticAnalyzer.registry.add_recognizer(firstName_recognizer)
staticAnalyzer.registry.add_recognizer(phone_recognizer)
staticAnalyzer.registry.add_recognizer(month_recognizer)
staticAnalyzer.registry.add_recognizer(country_recognizer)
staticAnalyzer.registry.add_recognizer(genericNumber_recognizer)
staticAnalyzer.registry.add_recognizer(state_recognizer)
staticAnalyzer.registry.add_recognizer(year_recognizer)
staticAnalyzer.registry.add_recognizer(overNinetyage_recognizer)

#prueba1, prueba2 = customAnonymizeAndExtract(text)

#print(prueba2)




In [4]:
import re

special_characters = r"[\.\+\*\?\|\(\)\[\]\{\}\\\^\$\#]"
def replace_secrets(sentence, secrets):
    pattern = re.compile("|".join(map(re.escape, secrets.keys())))
    return pattern.sub(lambda match: secrets[match.group(0)][0], sentence)

    
def customAnonymizeAndExtract(text: str) -> tuple[str, str]:
    dynamicEntities = []
    dynamicOperators = {}
    originalText = text
    textAnalysis = staticAnalyzer.analyze(text=text, language="en", entities=staticEntities)
    secrets = dict()
    secret_phrases = dict()
    
    for result in textAnalysis:
        rawSecret = text[result.start:result.end]
        regexAnalysis = staticAnalyzer.analyze(text=rawSecret, language="en", entities=staticEntities)
        fakedSecret = anonymizer.anonymize(rawSecret, regexAnalysis, staticOperators)
        if (secrets.get(rawSecret) == None):
            secretType = regexAnalysis[0].entity_type
            secrets[rawSecret] = fakedSecret.text
            secretRegex = re.escape(rawSecret)
            specific_pattern = Pattern(
            name=rawSecret,
            regex=secretRegex
            , 
            score=0.8
            )
            dynamicEntities.append(rawSecret)
            regex_recognizer = PatternRecognizer(
                supported_entity=rawSecret,
                patterns=[specific_pattern]
                )
            dynamicAnalyzer.registry.add_recognizer(regex_recognizer)
            substitution = fakedSecret.text + ""
            operatorLambda = OperatorConfig("custom", {"lambda": lambda text, sub=substitution: sub})
            dynamicOperators[rawSecret] = operatorLambda
            secret_phrases[secrets[rawSecret]] = {"repetitions": 1, "phrase":  fakedSecret.text, "type":secretType}
        else:
            secret_phrases[secrets[rawSecret]]["repetitions"] += 1

 
                    
    if (not secret_phrases):
        secret_phrases = None

    
    finalAnalysis = dynamicAnalyzer.analyze(text=originalText, language="en", entities=dynamicEntities) 
    anonymized_text =  anonymizer.anonymize(originalText, finalAnalysis, dynamicOperators)



    return (anonymized_text.text, secret_phrases)




In [5]:
import csv
from tqdm import tqdm
from itertools import islice
import json


def procesar_csv(input_csv, output_csv, secrets_csv, start_row=0):
    """
    Lee un CSV línea por línea desde una fila específica, aplica una transformación a cada string,
    y escribe el resultado en un nuevo archivo.
    """
    with open(input_csv, mode='r', encoding='utf-8') as infile, \
         open(output_csv, mode='a', encoding='utf-8', newline='') as outfile, \
         open(secrets_csv, mode='a', encoding='utf-8', newline='') as secretOutfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile)
        secretWriter = csv.writer(secretOutfile)
        secretWriter.writerow(['phrase', 'repetitions','type'])
        for txt in tqdm(islice(reader, start_row, None)):
            transformed_row, secret = customAnonymizeAndExtract(txt[10]) #Using TEXT column of MIMIC-III
            writer.writerow([transformed_row])
            if secret is not None:
                for s in secret:
                    secretWriter.writerow([secret[s]['phrase'], secret[s]['repetitions'], secret[s]['type']])
             

input_csv_path = "NOTEEVENTS_random.csv"
output_csv_path = "NOTEEVENTS_variablePresets_deanonymized.csv"
secret_csv_path = "secrets_variablePresets.txt"

procesar_csv(input_csv_path, output_csv_path, secret_csv_path, 10688)


890it [24:27,  1.65s/it]


KeyboardInterrupt: 

In [6]:
import csv
count = 0
with open('NOTEEVENTS_variablePresets_deanonymized.csv', newline='', encoding='ISO-8859-1') as csvfile:
          tweet_csv = csv.reader(csvfile)
          for row in tweet_csv:
                count+=1
                

In [7]:
count

11578